# Import and Shared functions

In [3]:
import pandas as pd
import numpy as np
import ephem
import math
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
from dateutil import parser as dateutil_parser
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', 'driver')))
import importlib
import kinematics
importlib.reload(kinematics)
from kinematics import calc_parallactic_angle, azaltroll_to_theta, apply_mechanical_corrections, azaltroll_to_q, MountModelParams
from control import theta_to_jacobian


In [21]:
def r2_score(y, yhat):
    ss_res = np.sum((y - yhat)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    return 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

def r2_quality(r2):
    return (
        "excellent" if r2>0.95 else 
        "good" if r2>0.8 else 
        "moderate" if r2>0.5 else 
        "weak" if r2>0.25 else 
        "poor")

def pec_quality_db(snr_db):
    return (
        "excellent" if snr_db > 30 else
        "good"      if snr_db > 20 else
        "moderate"  if snr_db > 10 else
        "weak"      if snr_db > 5  else
        "poor"
    )


In [22]:
import re
from quaternion import Q as Quaternion
import sys, os
log_path = '../logs/alpaca.sga.omega.log'   # adjust to your log file path

# ── Parse alignQ and roll_adj from log file ───────────────────────────────
def parse_quest_from_log(log_path):
    """
    Find the LAST QUEST Model entry before the first PECLOG entry.
    Returns alignQ (Quaternion) and roll_adj (degrees).
    """
    alignQ   = None
    roll_adj = 0.0
    last_q   = None
    last_r   = None

    quest_pattern = re.compile(
        r'QUEST Model.*w:\s*([+-]?\d+\.\d+).*x:\s*([+-]?\d+\.\d+)'
        r'.*y:\s*([+-]?\d+\.\d+).*z:\s*([+-]?\d+\.\d+)')
    
    roll_pattern = re.compile(
        r'Roll Adj:\s*([+-]?\d+)d(\d+)\'(\d+\.\d+)"')
    
    peclog_seen = False
    
    with open(log_path, encoding='utf-8', errors='replace') as f:
        for line in f:
            if 'PECLOG' in line:
                peclog_seen = True
                break   # stop at first PECLOG — use last QUEST before this
            
            m = quest_pattern.search(line)
            if m:
                last_q = Quaternion(
                    w=float(m.group(1)), x=float(m.group(2)),
                    y=float(m.group(3)), z=float(m.group(4)))
            
            m2 = roll_pattern.search(line)
            if m2:
                d = float(m2.group(1))
                mn = float(m2.group(2))
                s  = float(m2.group(3))
                sign = -1 if '-' in line[line.find('Roll Adj'):line.find('Roll Adj')+15] else 1
                last_r = sign * (abs(d) + mn/60 + s/3600)
    
    if last_q is not None:
        alignQ = last_q
    if last_r is not None:
        roll_adj = last_r
        
    return alignQ, roll_adj


alignQ, roll_adj = parse_quest_from_log(log_path)

print(f"Parsed alignQ: w={alignQ.w:.7f} x={alignQ.x:.7f} "
      f"y={alignQ.y:.7f} z={alignQ.z:.7f}")
print(f"Parsed roll_adj: {roll_adj:.5f}°  ({roll_adj*60:.3f}')")
print()

# ── MAC parameters — paste from fits_extract output ───────────────────────
mac = MountModelParams(
    m3_tilt_dm1      = -261.30,
    m3_tilt_dm2      = -148.69,
    m3_tilt_dm3      =    0.00,
    m2_tilt_dm2_amp  =  +94.78,
    m2_tilt_dm2_zero =  +20.88,
    m2_roll_coupling =    0.00,
    m2_roll_zero     =    0.00,
    m1_offset        =    0.00,
    m2_offset        =    0.00,
    m3_offset        =    0.00,
)

#Example Log of QUEST Model
#2026-04-26T07:53:54.246 INFO QUEST Model  | Points: 3 | w: -0.9956408 | x: -0.0035016 | y: +0.0057265 | z: +0.0930287 
#2026-04-26T07:53:54.246 INFO   MAC: ON    | LGC: ON   | RMS Residual: +000d26'41.60"  | Az Correction: +010d40'24.64" | Tilt: +000d46'08.91" @ +063d53'39.91" | Roll Adj: -000d08'19.95"


AttributeError: 'NoneType' object has no attribute 'w'

In [23]:
import pandas as pd
import numpy as np
from scipy import signal

# Parse log
cols = ['timestamp','zeta1','zeta2','zeta3','theta1','theta2','theta3',
        'az','alt','roll', 'phd2_ra','phd2_dec','phd2_pa']

rows = []
with open(log_path) as f:
    for line in f:
        if 'PECLOG' not in line:
            continue
        ts = line.split(' INFO ')[0].strip()
        vals = line.split('PECLOG,')[1].strip().split(',')
        rows.append([ts] + [float(v) for v in vals])

df = pd.DataFrame(rows, columns=cols)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['t_sec'] = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds()
df = df.sort_values('t_sec').reset_index(drop=True)

print(f"Duration: {df['t_sec'].iloc[-1]/3600:.2f} hours")
print(f"N samples: {len(df)}")
print(f"zeta1 range: {df['zeta1'].min():.3f} to {df['zeta1'].max():.3f}")
print()
df.columns


Duration: 2.23 hours
N samples: 15705
zeta1 range: -33.778 to -9.531



Index(['timestamp', 'zeta1', 'zeta2', 'zeta3', 'theta1', 'theta2', 'theta3',
       'az', 'alt', 'roll', 'phd2_ra', 'phd2_dec', 'phd2_pa', 't_sec'],
      dtype='object')

# Mount Orientation (Az, Alt, Roll) vs Time

In [24]:
xdata = df['t_sec'] / 60
ydata = [
    #row, dataset,            name,                color
    (1,   df['az'],    'Azimuth (°)',  'royalblue'),
    (2,   df['alt'],   'Altitude (°)', 'orange'),
    (3,   df['roll'],  'Roll (°)',     'mediumseagreen'),
]
rows = len(set([row for row, y,name,color in ydata]))
fig = make_subplots(rows=rows, cols=1, shared_xaxes=True,
    subplot_titles=[name for row, y,name,color in ydata],
    vertical_spacing=0.12)

for (row, y, name, color) in ydata:
    fig.add_trace(go.Scatter(x=xdata, y=y,
        mode='lines', line=dict(color=color, width=0.8),
        name=name), row=row, col=1)
    fig.update_yaxes(title_text=name, row=row, col=1)

fig.update_xaxes(title_text='Time (minutes)', row=3, col=1)
fig.update_layout(height=800, width=1200, template="plotly_dark",
        title='Range of Mount Orientation over session')
fig.show()

# Cumulative Pulse Guide Corrections vs Time

In [25]:
xdata = df['t_sec'] / 60
ydata = [
    #row, dataset,            name,                color
    (1,   df['phd2_ra']*60,  'PHD2 RA (arcmin)',  'royalblue'),
    (2,   df['phd2_dec']*60, 'PHD2 Dec (arcmin)', 'orange'),
    (3,   df['phd2_pa']*60,  'PHD2 PA (arcmin)',  'mediumseagreen'),
]
rows = len(set([row for row, y,name,color in ydata]))
fig = make_subplots(rows=rows, cols=1, shared_xaxes=True,
    subplot_titles=[name for row, y,name,color in ydata],
    vertical_spacing=0.12)


for (row, y, name, color) in ydata:
    fig.add_trace(go.Scatter(x=xdata, y=y,
        mode='lines', line=dict(color=color, width=0.8), hovertext=df['timestamp'],
        name=name), row=row, col=1)
    fig.update_yaxes(title_text=name, row=row, col=1)

fig.update_xaxes(title_text='Time (minutes)', row=3, col=1)
fig.update_layout(height=800, width=1200, template="plotly_dark",
        title='Cumulative Pulse Guide Corrections over session')
fig.show()

# Right Ascension Drift

In [39]:
t = df['t_sec'].values
dec = df['phd2_ra'].values
poly_coeffs, residuals, rank, sv, rcond = np.polyfit(t, dec, 1, full=True)
slope, intercept = poly_coeffs
dec_fit = np.polyval(poly_coeffs, t)
n = len(t)
rss = residuals[0] if len(residuals) > 0 else np.sum((dec - dec_fit)**2)
tss = np.sum((y - np.mean(dec))**2)
r2 = 1 - rss / tss if tss != 0 else float('nan')
rmse = np.sqrt(rss / n)
print(f"=== Right Ascension - Drift Fit Summary (N={n}) ===")
print(f"Slope (drift rate): {slope*60*60:+8.3f} arcmin/min")
print(f"Intercept:          {intercept*60:+8.3f} arcmin")
print(f"RMSE (noise):       {rmse*60:8.3f} arcmin")
print(f"R² (fit quality):   {r2:8.3f} {r2_quality(r2)} fit")

=== Right Ascension - Drift Fit Summary (N=15705) ===
Slope (drift rate):   -0.366 arcmin/min
Intercept:            +4.488 arcmin
RMSE (noise):          1.957 arcmin
R² (fit quality):      0.990 excellent fit


# Declination Drift

In [40]:
t = df['t_sec'].values
dec = df['phd2_dec'].values
poly_coeffs, residuals, rank, sv, rcond = np.polyfit(t, dec, 1, full=True)
slope, intercept = poly_coeffs
dec_fit = np.polyval(poly_coeffs, t)
n = len(t)
rss = residuals[0] if len(residuals) > 0 else np.sum((dec - dec_fit)**2)
tss = np.sum((y - np.mean(dec))**2)
r2 = 1 - rss / tss if tss != 0 else float('nan')
rmse = np.sqrt(rss / n)
print(f"=== Declination - Drift Fit Summary (N={n}) ===")
print(f"Slope (drift rate): {slope*60*60:+8.3f} arcmin/min")
print(f"Intercept:          {intercept*60:+8.3f} arcmin")
print(f"RMSE (noise):       {rmse*60:8.3f} arcmin")
print(f"R² (fit quality):   {r2:8.3f} {r2_quality(r2)} fit")



=== Declination - Drift Fit Summary (N=15705) ===
Slope (drift rate):   +0.271 arcmin/min
Intercept:           +13.399 arcmin
RMSE (noise):          2.889 arcmin
R² (fit quality):      0.992 excellent fit


# Right Ascension Period

In [45]:
from scipy.signal import periodogram
import numpy as np

# RA: the periodic PEC signal
ra = df['phd2_ra'].values

# Remove linear drift (same idea as DEC fit)
ra_detrended = ra - np.polyval(np.polyfit(t, ra, 1), t)

# Sampling frequency
fs = 1 / np.median(np.diff(t))

# Periodogram
freqs, power = periodogram(ra_detrended, fs=fs)

# Ignore zero frequency and periods > 100 min 
max_period_sec = 120 * 60
min_freq = 1 / max_period_sec
valid = (freqs >= min_freq)
freqs = freqs[valid]
power = power[valid]

# Peak detection
peak_idx = np.argmax(power)
peak_freq = freqs[peak_idx]
peak_power = power[peak_idx]

worm_period_sec = 1 / peak_freq

# Noise floor estimate (median is robust)
mask = np.ones_like(power, dtype=bool)
window = 3  # exclude ±3 bins around peak
mask[max(0, peak_idx-window):peak_idx+window+1] = False
noise_floor = np.median(power[mask])

# Signal-to-noise ratio
snr = peak_power / noise_floor if noise_floor > 0 else np.inf
snr_db = 10 * np.log10(peak_power / noise_floor)

# Estimate amplitude (rough, from detrended signal)
amp = (np.max(ra_detrended) - np.min(ra_detrended)) / 2

# Output
n = len(ra)

print(f"=== Right Ascension - PEC Analysis (N={n}) ===")
print(f"Worm period:        {worm_period_sec/60:8.3f} min")
print(f"Amplitude:          {amp*60:8.3f} arcmin")
print(f"Peak power:         {10*np.log10(peak_power):8.3f} dB (relative)")
print(f"Noise floor:        {10*np.log10(noise_floor):8.3f} dB (relative)")
print(f"SNR (periodicity):  {snr_db:8.2f} dB {pec_quality_db(snr_db)} signal")

=== Right Ascension - PEC Analysis (N=15705) ===
Worm period:          33.439 min
Amplitude:             4.580 arcmin
Peak power:            2.283 dB (relative)
Noise floor:         -63.855 dB (relative)
SNR (periodicity):     66.14 dB excellent signal


In [28]:
periods_min = 1 / freqs / 60
log_power = np.log10(power)
max_y = np.max(log_power)
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=periods_min,
    y=log_power,
    mode='lines',
    name='log10(Power)'
))

fig.add_vline(x=worm_period_sec/60, line_dash="dash", line_color="red")

fig.update_layout(
    title="RA Periodogram (Log Power)",
    xaxis_title="Period (minutes)",
    yaxis_title="log10(Power)", 
    height=600, width=1200, template="plotly_dark",
)
max_y
fig.update_yaxes(range=[ -4, max_y])

fig.show()

In [46]:
from scipy.signal import periodogram
import numpy as np

# Dec: the periodic PEC signal
dec = df['phd2_dec'].values

# Remove linear drift (same idea as DEC fit)
dec_detrended = dec - np.polyval(np.polyfit(t, dec, 1), t)

# Sampling frequency
fs = 1 / np.median(np.diff(t))

# Periodogram
freqs, power = periodogram(dec_detrended, fs=fs)

# Ignore zero frequency and periods > 100 min 
max_period_sec = 120 * 60
min_freq = 1 / max_period_sec
valid = (freqs >= min_freq)
freqs = freqs[valid]
power = power[valid]

# Peak detection
peak_idx = np.argmax(power)
peak_freq = freqs[peak_idx]
peak_power = power[peak_idx]

worm_period_sec = 1 / peak_freq

# Noise floor estimate (median is robust)
mask = np.ones_like(power, dtype=bool)
window = 3  # exclude ±3 bins around peak
mask[max(0, peak_idx-window):peak_idx+window+1] = False
noise_floor = np.median(power[mask])

# Signal-to-noise ratio
snr = peak_power / noise_floor if noise_floor > 0 else np.inf
snr_db = 10 * np.log10(peak_power / noise_floor)

# Estimate amplitude (rough, from detrended signal)
amp = (np.max(ra_detrended) - np.min(ra_detrended)) / 2

# Output
n = len(ra)

print(f"=== Declination - PEC Analysis (N={n}) ===")
print(f"Worm period:        {worm_period_sec/60:8.3f} min")
print(f"Amplitude:          {amp*60:8.3f} arcmin")
print(f"Peak power:         {10*np.log10(peak_power):8.3f} dB (relative)")
print(f"Noise floor:        {10*np.log10(noise_floor):8.3f} dB (relative)")
print(f"SNR (periodicity):  {snr_db:8.2f} dB {pec_quality_db(snr_db)} signal")

=== Declination - PEC Analysis (N=15705) ===
Worm period:          26.751 min
Amplitude:             4.580 arcmin
Peak power:            1.407 dB (relative)
Noise floor:         -65.508 dB (relative)
SNR (periodicity):     66.92 dB excellent signal


In [25]:
from scipy.optimize import curve_fit

def pec_model(t, period, *coeffs):
    n_harmonics = len(coeffs) // 2
    result = np.zeros_like(t, dtype=float)
    for k in range(n_harmonics):
        omega = 2 * np.pi * (k + 1) / period
        result += (
            coeffs[2*k] * np.sin(omega * t) +
            coeffs[2*k+1] * np.cos(omega * t)
        )
    return result

n_harmonics = 2
p0 = [0.0] * (2 * n_harmonics)
popt, _ = curve_fit(
    lambda t, *c: pec_model(t, worm_period_sec, *c),
    t, ra_detrended, p0=p0
)
ra_pec_fit = pec_model(t, worm_period_sec, *popt)

n = len(t)
residual = ra_detrended - ra_pec_fit
rss = np.sum(residual**2)
tss = np.sum((ra_detrended - np.mean(ra_detrended))**2)
r2 = 1 - rss / tss if tss != 0 else np.nan
rmse = np.sqrt(rss / n)

# Before/after noise comparison
baseline_rmse = np.sqrt(np.mean(ra_detrended**2))
reduction = 100 * (1 - rmse / baseline_rmse)

print(f"=== PEC Harmonic Fit Summary (N={n}) ===")
print(f"Worm period:        {worm_period_sec/60:8.2f} min")
print(f"Harmonics used:     {n_harmonics:8.0f}")

print(f"RMSE (after PEC):   {rmse*60:8.3f} arcmin")
print(f"RMSE (baseline):    {baseline_rmse*60:8.3f} arcmin")

print(f"Error reduction:    {reduction:8.2f} %")

print(f"R² (fit quality):   {r2:8.3f} {r2_quality(r2)} fit")

=== PEC Harmonic Fit Summary (N=12389) ===
Worm period:           34.90 min
Harmonics used:            2
RMSE (after PEC):      2.075 arcmin
RMSE (baseline):       3.519 arcmin
Error reduction:       41.05 %
R² (fit quality):      0.652 moderate fit


In [26]:
def get_pec_correction_arcmin(t_elapsed_sec: float, 
                               worm_period: float, 
                               coeffs: list) -> float:
    """Return RA correction to apply (arcmin)."""
    phase = t_elapsed_sec % worm_period
    return pec_model(np.array([phase]), worm_period, *coeffs)[0]

# Extract alignQ and roll_adj from log

# Remain

In [27]:
from scipy.signal import periodogram
# RA: the periodic PEC signal
ra = df['phd2_ra'].values
ra_detrended = ra - np.polyval(np.polyfit(t, ra, 1), t)

# Find worm period
fs = 1 / np.median(np.diff(t))  # sample rate in Hz
freqs, power = periodogram(ra_detrended, fs=fs)
peak_freq = freqs[np.argmax(power)]
worm_period_sec = 1 / peak_freq
print(f"Worm period: {worm_period_sec/60:.1f} minutes")

Worm period: 34.9 minutes


In [28]:

# ── Step 1: Remove long-term drift (polar misalignment trend) ─────────────
# Fit and subtract a low-order polynomial from phd2_ra_accum
# PE period ~568s is much shorter than the 4hr session
# so a 3rd order poly removes drift without touching PE signal
t = df['t_sec'].values
ra_accum = df['phd2_ra'].values * 3600  # convert to arcsec

poly_coeffs = np.polyfit(t, ra_accum, 3)
trend = np.polyval(poly_coeffs, t)
ra_detrended = ra_accum - trend

df['ra_detrended'] = ra_detrended

print("Detrended RA stats (arcsec):")
print(f"  std={ra_detrended.std():.2f}  "
      f"min={ra_detrended.min():.2f}  "
      f"max={ra_detrended.max():.2f}")
print()

Detrended RA stats (arcsec):
  std=164.76  min=-307.01  max=301.54



In [29]:


# ── Step 2: Phase fold to find worm period ────────────────────────────────
# zeta1 decreases as mount tracks west
# worm_period_deg = degrees of zeta1 per worm revolution
# Try a range of periods and find which gives lowest scatter in folded data

best_period = None
best_rms    = np.inf
results     = []

for period_deg in np.arange(0.5, 15.0, 0.05):
    phase = (df['zeta1'].values % period_deg) / period_deg  # [0,1)
    
    # Bin the detrended RA into phase bins
    n_bins   = 32
    bin_idx  = (phase * n_bins).astype(int) % n_bins
    bin_mean = np.array([ra_detrended[bin_idx == b].mean() 
                         if (bin_idx == b).sum() > 2 else np.nan 
                         for b in range(n_bins)])
    
    # RMS of residuals after subtracting binned mean
    residuals = ra_detrended - bin_mean[bin_idx]
    rms = np.sqrt(np.nanmean(residuals**2))
    results.append((period_deg, rms))
    
    if rms < best_rms:
        best_rms    = rms
        best_period = period_deg

results_df = pd.DataFrame(results, columns=['period_deg','rms'])
print(f"Best worm period: {best_period:.2f}° of zeta1")
print(f"Best RMS: {best_rms:.3f} arcsec")
print()

# Convert worm period from zeta1 degrees to time
# zeta1 rate = d(zeta1)/dt at each moment
dzeta1_dt = np.gradient(df['zeta1'].values, df['t_sec'].values)
mean_zeta1_rate = np.abs(np.mean(dzeta1_dt))  # deg/sec
worm_period_sec = best_period / mean_zeta1_rate
dt = np.diff(df['t_sec'].values)
dz = np.diff(df['zeta1'].values)
valid = dt > 0
mean_zeta1_rate = np.abs(np.median(dz[valid] / dt[valid]))  # deg/sec
worm_period_sec = best_period / mean_zeta1_rate

print(f"Mean zeta1 rate: {mean_zeta1_rate*3600:.4f} arcsec/sec")
print(f"Worm period in time: {worm_period_sec:.1f}s = {worm_period_sec/60:.2f} min")
print()

# ── Step 3: Build PE correction table ─────────────────────────────────────
n_bins  = 64
phase   = (df['zeta1'].values % best_period) / best_period
bin_idx = (phase * n_bins).astype(int) % n_bins

pe_table     = np.zeros(n_bins)
pe_table_std = np.zeros(n_bins)
pe_table_n   = np.zeros(n_bins)

for b in range(n_bins):
    mask = bin_idx == b
    if mask.sum() > 2:
        pe_table[b]     = np.mean(ra_detrended[mask])
        pe_table_std[b] = np.std(ra_detrended[mask])
        pe_table_n[b]   = mask.sum()

print("PE correction table (64 bins):")
print(f"  Amplitude: {pe_table.max() - pe_table.min():.1f} arcsec peak-to-peak")
print(f"  Mean bin std: {pe_table_std[pe_table_n>0].mean():.2f} arcsec")
print(f"  Coverage: {(pe_table_n>0).sum()}/{n_bins} bins populated")

Best worm period: 13.85° of zeta1
Best RMS: 38.602 arcsec

Mean zeta1 rate: 11.6719 arcsec/sec
Worm period in time: 4271.8s = 71.20 min

PE correction table (64 bins):
  Amplitude: 587.2 arcsec peak-to-peak
  Mean bin std: 23.18 arcsec
  Coverage: 64/64 bins populated


In [30]:

# ── Convert az/alt/roll → theta1/theta2/theta3 ────────────────────────────
# Using azaltroll_to_theta which already handles the FK geometry.
# This gives us the corrected motor angles consistent with the driver.
# Note: azaltroll_to_theta is the IK path without QUEST/MAC,
# so we pass the plate-solved az/alt/roll which already reflects
# the true sky position — theta1/2/3 here are the ideal motor angles.

def compute_theta(row):
    t1, t2, t3 = azaltroll_to_theta(row['az'], row['alt'], row['rollAngle'])
    return pd.Series({'theta1': t1, 'theta2': t2, 'theta3': t3})

print("Computing theta1/theta2/theta3 from az/alt/roll...")
theta_df = df.apply(compute_theta, axis=1)
df = pd.concat([df, theta_df], axis=1)

print("theta stats:")
for col in ['theta1','theta2','theta3']:
    print(f"  {col}: {df[col].min():.2f}° to {df[col].max():.2f}°")
print()

# ── Full Jacobian projection of PHD2 corrections → motor space ────────────
def phd2_to_motor_jacobian(df, alignQ, roll_adj, mac, lat_deg=-33.6554):
    """
    Convert PHD2 RA/Dec sky corrections to M1/M2/M3 motor corrections
    using the full Jacobian at each timestep.
    
    Steps:
    1. Per-step PHD2 correction in RA/Dec degrees
    2. Convert to sky arc (applying cos(dec) to RA)
    3. Rotate from equatorial to horizon frame via parallactic angle
    4. Express as angular velocity vector in mount base frame
    5. Solve J^-1 * omega → motor angle corrections
    """
    import math

    
    lat = math.radians(lat_deg)
    
    # Per-step deltas of accumulated corrections
    ra_accum  = df['phd2_ra_accum'].values
    dec_accum = df['phd2_dec_accum'].values
    dra_deg   = np.diff(ra_accum,  prepend=ra_accum[0])
    ddec_deg  = np.diff(dec_accum, prepend=dec_accum[0])
    
    # Sky arc arcsec
    dec_rad     = np.radians(df['dec'].values)
    dra_sky_as  = dra_deg  * np.cos(dec_rad) * 3600
    ddec_sky_as = ddec_deg * 3600
    
    az_deg  = df['az'].values
    alt_deg = df['alt'].values
    t1      = df['theta1'].values
    t2      = df['theta2'].values
    t3      = df['theta3'].values
    
    dM1 = np.zeros(len(df))
    dM2 = np.zeros(len(df))
    dM3 = np.zeros(len(df))
    
    for i in range(len(df)):
        if abs(dra_sky_as[i]) < 1e-9 and abs(ddec_sky_as[i]) < 1e-9:
            continue
        
        az  = math.radians(az_deg[i])
        alt = math.radians(alt_deg[i])
        
        # Parallactic angle — using same convention as kinematics.py
        num = math.sin(az)
        den = math.tan(lat)*math.cos(alt) - math.sin(alt)*math.cos(az)
        pa  = math.degrees(math.atan2(num, den))
        pa  = ((- pa + 180) % 360) - 180   # wrap180(-angle) per kinematics
        pa_rad = math.radians(pa)
        
        # Rotate equatorial → horizon frame
        # [dAz_sky ]   [cos(pa)   sin(pa)] [dRA_sky ]
        # [dAlt_sky] = [-sin(pa)  cos(pa)] [dDec_sky]
        daz_as  = ( math.cos(pa_rad)*dra_sky_as[i] 
                  + math.sin(pa_rad)*ddec_sky_as[i])
        dalt_as = (-math.sin(pa_rad)*dra_sky_as[i] 
                  + math.cos(pa_rad)*ddec_sky_as[i])
        
        # Convert arcsec → degrees
        daz_deg_  = daz_as  / 3600
        dalt_deg_ = dalt_as / 3600
        
        # Build angular velocity vector in base frame (radians)
        # M1 axis: world vertical [0,0,1]
        # M2 axis: horizontal after az rotation
        # Correction in az maps to world Z rotation / cos(alt)
        # Correction in alt maps to M2 axis rotation
        from quaternion import Q as Quaternion as Q
        qtheta1 = Q(axis=[0,0,1], degrees=-t1[i]+90)
        a2 = np.array(qtheta1.rotate([0,1,0]))   # M2 axis in base frame
        
        omega = (np.array([0.,0.,1.]) * math.radians(daz_deg_ / math.cos(alt))
               + a2                   * math.radians(dalt_deg_))
        
        try:
            J         = theta_to_jacobian(t1[i], t2[i], t3[i])
            theta_dot = np.linalg.solve(J, omega)   # radians per step
            dM1[i]    = math.degrees(theta_dot[0])
            dM2[i]    = math.degrees(theta_dot[1])
            dM3[i]    = math.degrees(theta_dot[2])
        except np.linalg.LinAlgError:
            pass
    
    df['dM1'] = dM1
    df['dM2'] = dM2
    df['dM3'] = dM3
    df['M1_corr_arcsec'] = np.cumsum(dM1) * 3600
    df['M2_corr_arcsec'] = np.cumsum(dM2) * 3600
    df['M3_corr_arcsec'] = np.cumsum(dM3) * 3600
    return df

df = phd2_to_motor_jacobian(df, alignQ, roll_adj, mac)

# ── Plot ──────────────────────────────────────────────────────────────────
fig = make_subplots(rows=3, cols=1,
    subplot_titles=[
        'M1 (Az motor) cumulative correction — M1 worm PE',
        'M2 (Alt motor) cumulative correction — M2 worm PE',
        'M3 (Roll motor) cumulative correction — M3 worm PE'],
    vertical_spacing=0.06)

t_hrs = df['t_sec'] / 3600
colors = ['royalblue','orange','mediumseagreen']
for row, (col, color) in enumerate(zip(
        ['M1_corr_arcsec','M2_corr_arcsec','M3_corr_arcsec'], colors), 1):
    fig.add_trace(go.Scatter(
        x=t_hrs, y=df[col],
        mode='lines', line=dict(color=color, width=0.8),
        name=col), row=row, col=1)

fig.update_yaxes(title_text='arcsec')
fig.update_xaxes(title_text='Time (hours)', row=3, col=1)
fig.update_layout(
    height=750, width=1200,
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117',
    font=dict(color='#aaaacc'),
    title='PHD2 corrections → M1/M2/M3 via full Jacobian')
fig.show()

print("\nMotor correction totals over session:")
for ax in ['M1','M2','M3']:
    total = df[f'{ax}_corr_arcsec'].iloc[-1]
    rate  = total / (df['t_sec'].iloc[-1]/3600)
    print(f"  {ax}: {total:+.1f} arcsec total  ({rate:+.1f} arcsec/hour)")

SyntaxError: invalid syntax (4039682658.py, line 91)

In [ ]:
# ── Detect discontinuities in accumulated corrections ─────────────────────
# Both dithers and star reacquisitions appear as sudden jumps in the 
# per-step delta. The difference is magnitude:
#   Normal guide correction : < 5 arcsec/step
#   Dither                  : 10-100 arcsec/step  
#   Star reacquisition      : could be anything — often 100s of arcsec

ra_accum  = df['phd2_ra_accum'].values * 3600   # arcsec
dec_accum = df['phd2_dec_accum'].values * 3600

dra  = np.diff(ra_accum,  prepend=ra_accum[0])
ddec = np.diff(dec_accum, prepend=dec_accum[0])

# Total sky correction magnitude per step
step_mag = np.sqrt(dra**2 + ddec**2)

# Show distribution to find natural threshold
percentiles = [50, 75, 90, 95, 99, 99.9]
print("Step magnitude distribution (arcsec):")
for p in percentiles:
    print(f"  {p:5.1f}th percentile: {np.percentile(step_mag, p):.4f}")
print(f"  max: {step_mag.max():.4f}")
print()

# Plot step magnitude over time to visually identify events
fig = make_subplots(rows=2, cols=1,
    subplot_titles=['Step magnitude (arcsec) — log scale',
                    'Accumulated RA correction (arcsec)'],
    vertical_spacing=0.1)

t_hrs = df['t_sec'] / 3600

fig.add_trace(go.Scatter(
    x=t_hrs, y=step_mag,
    mode='lines', line=dict(color='orange', width=0.5),
    name='step magnitude'), row=1, col=1)

fig.add_trace(go.Scatter(
    x=t_hrs, y=ra_accum,
    mode='lines', line=dict(color='royalblue', width=0.8),
    name='RA accum'), row=2, col=1)

fig.update_yaxes(type='log', row=1, col=1, title_text='arcsec (log)')
fig.update_yaxes(title_text='arcsec', row=2, col=1)
fig.update_xaxes(title_text='Time (hours)', row=2, col=1)
fig.update_layout(height=600, width=1200,
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117',
    font=dict(color='#aaaacc'),
    title='Step magnitude — identify dithers and star reacquisitions')
fig.show()

# ── Clean accumulated corrections by removing all large jumps ─────────────
def remove_jumps(accum_arcsec, threshold_arcsec):
    """
    Remove all jumps larger than threshold from accumulated signal.
    Each jump is subtracted from all subsequent values.
    Returns cleaned signal and list of (index, jump_size) tuples.
    """
    delta  = np.diff(accum_arcsec, prepend=accum_arcsec[0])
    jumps  = []
    clean  = accum_arcsec.copy().astype(float)
    offset = 0.0
    
    for i in range(1, len(delta)):
        if abs(delta[i]) > threshold_arcsec:
            jumps.append((i, delta[i]))
            offset -= delta[i]
        clean[i] += offset
    
    return clean, jumps

# Use log-scale plot to pick threshold — try auto from percentiles
# Normal steps should be < 99th percentile, events are outliers
threshold = np.percentile(step_mag, 99.5)
print(f"Auto threshold (99.5th percentile): {threshold:.3f} arcsec")
print()

ra_clean,  ra_jumps  = remove_jumps(ra_accum,  threshold)
dec_clean, dec_jumps = remove_jumps(dec_accum, threshold)

print(f"Events removed from RA:  {len(ra_jumps)}")
for idx, jump in ra_jumps[:20]:
    print(f"  t={df['t_sec'].iloc[idx]/3600:.3f}h  jump={jump:+.2f}\"  "
          f"az={df['az'].iloc[idx]:.1f}°  alt={df['alt'].iloc[idx]:.1f}°")

print(f"\nEvents removed from Dec: {len(dec_jumps)}")
for idx, jump in dec_jumps[:20]:
    print(f"  t={df['t_sec'].iloc[idx]/3600:.3f}h  jump={jump:+.2f}\"  "
          f"az={df['az'].iloc[idx]:.1f}°  alt={df['alt'].iloc[idx]:.1f}°")

# Store cleaned versions
df['ra_clean_arcsec']  = ra_clean
df['dec_clean_arcsec'] = dec_clean

# Plot before/after
fig2 = make_subplots(rows=2, cols=1,
    subplot_titles=['RA accumulated — before and after jump removal',
                    'Dec accumulated — before and after jump removal'],
    vertical_spacing=0.1)

fig2.add_trace(go.Scatter(x=t_hrs, y=ra_accum,
    mode='lines', line=dict(color='rgba(100,150,255,0.4)', width=0.8),
    name='RA raw'), row=1, col=1)
fig2.add_trace(go.Scatter(x=t_hrs, y=ra_clean,
    mode='lines', line=dict(color='royalblue', width=0.8),
    name='RA cleaned'), row=1, col=1)

fig2.add_trace(go.Scatter(x=t_hrs, y=dec_accum,
    mode='lines', line=dict(color='rgba(255,180,100,0.4)', width=0.8),
    name='Dec raw'), row=2, col=1)
fig2.add_trace(go.Scatter(x=t_hrs, y=dec_clean,
    mode='lines', line=dict(color='orange', width=0.8),
    name='Dec cleaned'), row=2, col=1)

# Mark jump locations
for idx, _ in ra_jumps:
    fig2.add_vline(x=df['t_sec'].iloc[idx]/3600,
                   line=dict(color='red', width=1, dash='dot'), row=1, col=1)
for idx, _ in dec_jumps:
    fig2.add_vline(x=df['t_sec'].iloc[idx]/3600,
                   line=dict(color='red', width=1, dash='dot'), row=2, col=1)

fig2.update_yaxes(title_text='arcsec')
fig2.update_xaxes(title_text='Time (hours)', row=2, col=1)
fig2.update_layout(height=600, width=1200,
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117',
    font=dict(color='#aaaacc'),
    title='Jump removal — dithers and star reacquisitions')
fig2.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=3, cols=1, 
                    subplot_titles=['Raw accumulated PHD2 RA correction',
                                   'Detrended RA — should show PE sinusoid',
                                   'zeta1 vs time — should be monotonically decreasing'],
                    vertical_spacing=0.08)

t_hrs = df['t_sec'] / 3600

# Panel 1: raw accumulated RA
fig.add_trace(go.Scatter(
    x=t_hrs, y=df['phd2_ra_accum'] * 3600,
    mode='lines', line=dict(color='royalblue', width=0.8),
    name='raw RA accum'), row=1, col=1)

# Panel 2: detrended RA
fig.add_trace(go.Scatter(
    x=t_hrs, y=df['ra_detrended'],
    mode='lines', line=dict(color='royalblue', width=0.8),
    name='detrended RA'), row=2, col=1)
fig.add_hline(y=0, line=dict(color='red', width=1, dash='dot'), row=2, col=1)

# Panel 3: zeta1
fig.add_trace(go.Scatter(
    x=t_hrs, y=df['zeta1'],
    mode='lines', line=dict(color='mediumseagreen', width=0.8),
    name='zeta1'), row=3, col=1)

# Mark large jumps in RA accum
dra = np.diff(df['phd2_ra_accum'].values * 3600)
jumps = np.where(np.abs(dra) > 10)[0]
if len(jumps) > 0:
    fig.add_trace(go.Scatter(
        x=t_hrs.iloc[jumps],
        y=(df['phd2_ra_accum'].iloc[jumps] * 3600),
        mode='markers',
        marker=dict(color='red', size=8, symbol='x'),
        name='large jumps'), row=1, col=1)

fig.update_xaxes(title_text='Time (hours)')
fig.update_yaxes(title_text='arcsec', row=1, col=1)
fig.update_yaxes(title_text='arcsec', row=2, col=1)
fig.update_yaxes(title_text='degrees', row=3, col=1)

fig.update_layout(
    height=900, width=1200,
    plot_bgcolor='#0d1117',
    paper_bgcolor='#0d1117',
    font=dict(color='#aaaacc'),
    title='PEC Diagnostic — Raw, Detrended, zeta1',
    showlegend=True)

fig.show()

# Print jump stats to notebook
print(f"Large jumps in RA accum (>10\"/step): {len(jumps)}")
if len(jumps) > 0:
    print("First 20 jump locations:")
    for j in jumps[:20]:
        print(f"  t={df['t_sec'].iloc[j]/3600:.3f}h  "
              f"dra={dra[j]:+.2f}\"  "
              f"zeta1={df['zeta1'].iloc[j]:.4f}°")

resets = np.where(dra < -50)[0]
print(f"\nPossible PHD2 resets (RA drops >50\"): {len(resets)}")
for r in resets[:10]:
    print(f"  t={df['t_sec'].iloc[r]/3600:.3f}h  "
          f"ra_before={df['phd2_ra_accum'].iloc[r]*3600:.1f}\"  "
          f"ra_after={df['phd2_ra_accum'].iloc[r+1]*3600:.1f}\"")

In [ ]:
# ── Step 1: Remove dither jumps ───────────────────────────────────────────
ra_arcsec = df['phd2_ra_accum'].values * 3600

# Compute per-step delta
dra = np.diff(ra_arcsec)

# Dithers are large instantaneous jumps — much larger than any single
# guide correction. At 1Hz logging and typical guide rates, a single
# guide step moves <5 arcsec. Dithers move 10-100 arcsec instantly.
dither_threshold = 8.0   # arcsec — tune if needed

# Find dither indices
dither_idx = np.where(np.abs(dra) > dither_threshold)[0]
print(f"Dither events detected: {len(dither_idx)}")

# Remove dither steps by subtracting cumulative dither offsets
# Build a correction array: at each dither, subtract the jump
dither_correction = np.zeros(len(ra_arcsec))
for idx in dither_idx:
    dither_correction[idx+1:] -= dra[idx]   # remove the jump from here onwards

ra_dither_removed = ra_arcsec + dither_correction

# ── Step 2: Now detrend the dither-cleaned signal ─────────────────────────
t = df['t_sec'].values
poly_coeffs  = np.polyfit(t, ra_dither_removed, 3)
trend        = np.polyval(poly_coeffs, t)
ra_detrended = ra_dither_removed - trend

df['ra_dither_removed'] = ra_dither_removed
df['ra_detrended']      = ra_detrended

print(f"Detrended RA after dither removal:")
print(f"  std={ra_detrended.std():.2f}\"  "
      f"min={ra_detrended.min():.2f}\"  "
      f"max={ra_detrended.max():.2f}\"")

# ── Step 3: Diagnostic plot ───────────────────────────────────────────────
fig = make_subplots(rows=3, cols=1,
                    subplot_titles=[
                        'RA after dither removal (arcsec)',
                        'Detrended RA — PE signal',
                        'zeta1 vs time'],
                    vertical_spacing=0.08)

t_hrs = df['t_sec'] / 3600

fig.add_trace(go.Scatter(x=t_hrs, y=ra_dither_removed,
    mode='lines', line=dict(color='royalblue', width=0.8),
    name='dither removed'), row=1, col=1)

fig.add_trace(go.Scatter(x=t_hrs, y=ra_detrended,
    mode='lines', line=dict(color='royalblue', width=0.8),
    name='detrended'), row=2, col=1)
fig.add_hline(y=0, line=dict(color='red', width=1, dash='dot'), row=2, col=1)

fig.add_trace(go.Scatter(x=t_hrs, y=df['zeta1'],
    mode='lines', line=dict(color='mediumseagreen', width=0.8),
    name='zeta1'), row=3, col=1)

# Mark remaining jumps after dither removal
dra2   = np.diff(ra_dither_removed)
jumps2 = np.where(np.abs(dra2) > dither_threshold)[0]
if len(jumps2) > 0:
    fig.add_trace(go.Scatter(
        x=t_hrs.iloc[jumps2],
        y=ra_dither_removed[jumps2],
        mode='markers',
        marker=dict(color='red', size=8, symbol='x'),
        name='remaining jumps'), row=1, col=1)

fig.update_xaxes(title_text='Time (hours)')
fig.update_yaxes(title_text='arcsec', row=1, col=1)
fig.update_yaxes(title_text='arcsec', row=2, col=1)
fig.update_yaxes(title_text='degrees', row=3, col=1)
fig.update_layout(
    height=900, width=1200,
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117',
    font=dict(color='#aaaacc'),
    title='PEC Diagnostic — After Dither Removal')
fig.show()

In [ ]:
# ── Fix 1: Correct dither removal ─────────────────────────────────────────
ra_arcsec = df['phd2_ra_accum'].values * 3600
dra = np.diff(ra_arcsec)

# Find dithers — large instantaneous jumps
# Look at the distribution of step sizes to set threshold automatically
step_std = np.std(dra)
step_med = np.median(np.abs(dra))
dither_threshold = max(8.0, step_med + 5 * step_std)
print(f"Step median: {step_med:.3f}\"/s  std: {step_std:.3f}\"/s")
print(f"Auto dither threshold: {dither_threshold:.2f}\"")

dither_idx  = np.where(np.abs(dra) > dither_threshold)[0]
print(f"Dither events: {len(dither_idx)}")
print(f"Dither sizes (arcsec): min={np.abs(dra[dither_idx]).min():.1f}  "
      f"max={np.abs(dra[dither_idx]).max():.1f}  "
      f"mean={np.abs(dra[dither_idx]).mean():.1f}")

# Subtract cumulative dither offset
ra_clean = ra_arcsec.copy().astype(float)
cumulative_offset = 0.0
for idx in dither_idx:
    cumulative_offset += dra[idx]
    ra_clean[idx+1:] -= dra[idx]

print(f"\nAfter dither removal:")
print(f"  range: {ra_clean.min():.1f} to {ra_clean.max():.1f} arcsec")
print(f"  total drift: {ra_clean[-1] - ra_clean[0]:.1f} arcsec over "
      f"{df['t_sec'].iloc[-1]/3600:.2f} hours")
print(f"  drift rate: {(ra_clean[-1]-ra_clean[0])/df['t_sec'].iloc[-1]*3600:.1f} arcsec/hour")

# ── Fix 2: Detrend with linear fit (polar drift should be linear) ──────────
t = df['t_sec'].values
# Use linear fit — polar drift is constant rate
lin_coeffs   = np.polyfit(t, ra_clean, 1)
lin_trend    = np.polyval(lin_coeffs, t)
ra_detrended = ra_clean - lin_trend

df['ra_clean']     = ra_clean
df['ra_detrended'] = ra_detrended

print(f"\nDetrended stats:")
print(f"  std={ra_detrended.std():.2f}\"  "
      f"min={ra_detrended.min():.2f}\"  "
      f"max={ra_detrended.max():.2f}\"")

# ── Fix 3: Measure PE period directly from zero crossings ─────────────────
# Find zero crossings of the detrended signal
from scipy.signal import savgol_filter

# Smooth lightly to remove noise before finding zero crossings
ra_smooth = savgol_filter(ra_detrended, window_length=31, polyorder=3)

# Find positive-going zero crossings
signs     = np.sign(ra_smooth)
crossings = np.where((signs[:-1] < 0) & (signs[1:] >= 0))[0]
print(f"\nPositive zero crossings: {len(crossings)}")
if len(crossings) > 1:
    crossing_times = df['t_sec'].values[crossings]
    periods = np.diff(crossing_times)
    print(f"Period estimates (seconds): {periods.round(1)}")
    print(f"Mean period: {np.mean(periods):.1f}s = {np.mean(periods)/60:.2f} min")
    print(f"Std: {np.std(periods):.1f}s")

# ── Fix 4: Period search in zeta1 space ───────────────────────────────────
print(f"\nPhase fold search in zeta1 space:")
best_period = None
best_rms    = np.inf
results     = []

# zeta1 total span
z1_span = df['zeta1'].max() - df['zeta1'].min()
print(f"zeta1 total span: {z1_span:.2f}°")
print(f"Searching periods from 0.5° to {z1_span/3:.1f}°")

for period_deg in np.arange(0.5, z1_span/3, 0.02):
    phase   = (df['zeta1'].values % period_deg) / period_deg
    n_bins  = 32
    bin_idx = (phase * n_bins).astype(int) % n_bins
    bin_mean = np.array([
        ra_detrended[bin_idx == b].mean() if (bin_idx == b).sum() > 2 
        else np.nan for b in range(n_bins)])
    valid = ~np.isnan(bin_mean)
    if valid.sum() < 20:
        continue
    residuals = ra_detrended - np.where(~np.isnan(bin_mean[bin_idx]),
                                         bin_mean[bin_idx], 0)
    rms = np.sqrt(np.nanmean(residuals**2))
    results.append((period_deg, rms))
    if rms < best_rms:
        best_rms    = rms
        best_period = period_deg

results_df = pd.DataFrame(results, columns=['period_deg', 'rms'])

# Convert best period to time
dt   = np.diff(df['t_sec'].values)
dz   = np.diff(df['zeta1'].values)
valid = dt > 0
z1_rate = np.abs(np.median(dz[valid] / dt[valid]))   # deg/sec
worm_period_sec = best_period / z1_rate

print(f"\nBest worm period: {best_period:.3f}° of zeta1")
print(f"zeta1 rate: {z1_rate*3600:.4f} arcsec/sec = {z1_rate:.6f} deg/sec")
print(f"Worm period in time: {worm_period_sec:.1f}s = {worm_period_sec/60:.2f} min")
print(f"Best RMS: {best_rms:.2f} arcsec")

# ── Plot results ───────────────────────────────────────────────────────────
fig = make_subplots(rows=4, cols=1,
    subplot_titles=[
        'RA after dither removal',
        'Detrended RA — PE signal (linear drift removed)',
        'Phase fold RMS vs period (lower = better period match)',
        f'PE waveform — folded at {best_period:.2f}° of zeta1'],
    vertical_spacing=0.07)

t_hrs = df['t_sec'] / 3600

fig.add_trace(go.Scatter(x=t_hrs, y=ra_clean,
    mode='lines', line=dict(color='royalblue', width=0.8),
    name='dither removed'), row=1, col=1)

fig.add_trace(go.Scatter(x=t_hrs, y=ra_detrended,
    mode='lines', line=dict(color='royalblue', width=0.8),
    name='detrended'), row=2, col=1)
fig.add_hline(y=0, line=dict(color='red', width=1, dash='dot'), row=2, col=1)

# Zero crossings on detrended
if len(crossings) > 0:
    fig.add_trace(go.Scatter(
        x=t_hrs.values[crossings],
        y=np.zeros(len(crossings)),
        mode='markers', marker=dict(color='yellow', size=8),
        name='zero crossings'), row=2, col=1)

# Period search
fig.add_trace(go.Scatter(
    x=results_df['period_deg'], y=results_df['rms'],
    mode='lines', line=dict(color='orange', width=1),
    name='fold RMS'), row=3, col=1)
fig.add_vline(x=best_period, line=dict(color='red', dash='dot'), row=3, col=1)

# Folded PE waveform
phase_best = (df['zeta1'].values % best_period) / best_period
n_bins     = 64
bin_edges  = np.linspace(0, 1, n_bins+1)
bin_cents  = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_idx    = (phase_best * n_bins).astype(int) % n_bins
pe_table   = np.array([ra_detrended[bin_idx==b].mean() 
                        if (bin_idx==b).sum()>2 else np.nan 
                        for b in range(n_bins)])
pe_std     = np.array([ra_detrended[bin_idx==b].std()
                        if (bin_idx==b).sum()>2 else np.nan
                        for b in range(n_bins)])

fig.add_trace(go.Scatter(
    x=bin_cents, y=pe_table,
    mode='lines+markers', 
    line=dict(color='mediumseagreen', width=2),
    name='PE waveform'), row=4, col=1)
fig.add_trace(go.Scatter(
    x=np.concatenate([bin_cents, bin_cents[::-1]]),
    y=np.concatenate([pe_table+pe_std, (pe_table-pe_std)[::-1]]),
    fill='toself', fillcolor='rgba(50,200,100,0.15)',
    line=dict(width=0), name='±1σ'), row=4, col=1)
fig.add_hline(y=0, line=dict(color='red', width=1, dash='dot'), row=4, col=1)

fig.update_xaxes(title_text='Time (hours)', row=1, col=1)
fig.update_xaxes(title_text='Time (hours)', row=2, col=1)
fig.update_xaxes(title_text='Period (° of zeta1)', row=3, col=1)
fig.update_xaxes(title_text='Phase (0-1 of worm cycle)', row=4, col=1)
fig.update_yaxes(title_text='arcsec', row=1, col=1)
fig.update_yaxes(title_text='arcsec', row=2, col=1)
fig.update_yaxes(title_text='RMS (arcsec)', row=3, col=1)
fig.update_yaxes(title_text='PE correction (arcsec)', row=4, col=1)

fig.update_layout(
    height=1100, width=1200,
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117',
    font=dict(color='#aaaacc'),
    title='PEC Analysis — Dither Removed, Detrended, Period Search, Waveform')
fig.show()

In [ ]:
import math

def phd2_to_motor_corrections(df, guide_rate_ra_dps, guide_rate_dec_dps):
    """
    Convert accumulated PHD2 RA/Dec corrections to M1/M2 motor space.
    
    PHD2 sends corrections in equatorial frame (RA/Dec).
    We need to project these onto the mount's motor axes.
    
    At each timestep:
    - A RA correction of δRA degrees moves the boresight by δRA*cos(dec) 
      arcsec in the RA direction
    - This maps to motor corrections via the inverse Jacobian
    """
    lat = math.radians(-33.6554)
    
    # Per-step corrections (diff of cumulative)
    ra_accum_deg  = df['phd2_ra_accum'].values   # degrees
    dec_accum_deg = df['phd2_dec_accum'].values  # degrees
    
    dra_deg  = np.diff(ra_accum_deg,  prepend=ra_accum_deg[0])
    ddec_deg = np.diff(dec_accum_deg, prepend=dec_accum_deg[0])
    
    # Convert to sky arcsec motion
    # RA: delta_ra_deg * cos(dec) * 3600 = sky arcsec in RA direction
    # Dec: delta_dec_deg * 3600 = sky arcsec in Dec direction
    dec_rad = np.radians(df['dec'].values)
    dra_sky_arcsec  = dra_deg  * np.cos(dec_rad) * 3600
    ddec_sky_arcsec = ddec_deg * 3600

    # Convert sky RA/Dec motion to az/alt motion
    # At current az/alt, the transformation is:
    # dAz  =  dRA * cos(dec) / cos(alt)  ... approximately
    # dAlt =  dRA * sin(dec)*sin(lat)/cos(alt) + dDec*cos(lat)
    # But more precisely use the parallactic angle rotation:
    az_rad  = np.radians(df['az'].values)
    alt_rad = np.radians(df['alt'].values)
    
    # Parallactic angle (rotation from equatorial to horizon frame)
    def pa_rad(az, alt, lat):
        num = np.sin(az)
        den = np.tan(lat) * np.cos(alt) - np.sin(alt) * np.cos(az)
        return np.arctan2(num, den)
    
    pa = pa_rad(az_rad, alt_rad, lat)
    
    # Rotate RA/Dec corrections by parallactic angle to get az/alt corrections
    # [dAz ]   [ cos(pa)  sin(pa)] [dRA_sky ]
    # [dAlt] = [-sin(pa)  cos(pa)] [dDec_sky]
    daz_arcsec  =   np.cos(pa) * dra_sky_arcsec + np.sin(pa) * ddec_sky_arcsec
    dalt_arcsec =  -np.sin(pa) * dra_sky_arcsec + np.cos(pa) * ddec_sky_arcsec
    
    # Convert az/alt arcsec to M1/M2 motor degrees
    # At theta3≈0: dM1 ≈ dAz/cos(alt),  dM2 ≈ dAlt
    # More precisely use Jacobian inverse, but at theta3≈0 this is good
    theta3_rad = np.radians(df['theta3'].values)
    
    # M1 and M2 corrections in motor degrees
    dM1_deg = (daz_arcsec / 3600) / np.cos(alt_rad)  # az motor
    dM2_deg = (dalt_arcsec / 3600)                    # alt motor
    
    # Cumulative motor corrections
    df['dM1_step'] = dM1_deg
    df['dM2_step'] = dM2_deg
    df['M1_correction_deg']  = np.cumsum(dM1_deg)
    df['M2_correction_deg']  = np.cumsum(dM2_deg)
    df['M1_correction_arcsec'] = df['M1_correction_deg'] * 3600
    df['M2_correction_arcsec'] = df['M2_correction_deg'] * 3600
    
    return df

df = phd2_to_motor_corrections(df, 
     guide_rate_ra_dps=0.75 * 15/3600,
     guide_rate_dec_dps=0.75 * 15/3600)

print("Motor correction stats:")
print(f"  M1 total: {df['M1_correction_arcsec'].iloc[-1]:.1f} arcsec")
print(f"  M2 total: {df['M2_correction_arcsec'].iloc[-1]:.1f} arcsec")
print()

# Now phase fold M1 and M2 corrections separately
# M1 PE → fold against zeta1
# M2 PE → fold against zeta2

fig = make_subplots(rows=2, cols=1,
    subplot_titles=['M1 (Az motor) cumulative correction — M1 worm PE',
                    'M2 (Alt motor) cumulative correction — M2 worm PE'])

t_hrs = df['t_sec'] / 3600

fig.add_trace(go.Scatter(x=t_hrs, y=df['M1_correction_arcsec'],
    mode='lines', line=dict(color='royalblue', width=0.8),
    name='M1 correction'), row=1, col=1)

fig.add_trace(go.Scatter(x=t_hrs, y=df['M2_correction_arcsec'],
    mode='lines', line=dict(color='orange', width=0.8),
    name='M2 correction'), row=2, col=1)

fig.update_yaxes(title_text='arcsec', row=1, col=1)
fig.update_yaxes(title_text='arcsec', row=2, col=1)
fig.update_xaxes(title_text='Time (hours)', row=2, col=1)
fig.update_layout(
    height=600, width=1200,
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117',
    font=dict(color='#aaaacc'),
    title='PHD2 corrections projected to M1/M2 motor space')
fig.show()

In [ ]:
# Verify the parallactic angle during the session
print("Parallactic angle stats during session:")
pa_deg = np.degrees(np.arctan2(
    np.sin(np.radians(df['az'].values)),
    np.tan(np.radians(-33.6554)) * np.cos(np.radians(df['alt'].values)) - 
    np.sin(np.radians(df['alt'].values)) * np.cos(np.radians(df['az'].values))
))
# Apply wrap
pa_deg = ((- pa_deg + 180) % 360) - 180

print(f"  PA range: {pa_deg.min():.1f}° to {pa_deg.max():.1f}°")
print(f"  PA at t=0: {pa_deg[0]:.1f}°")
print(f"  PA at meridian transit: {pa_deg[np.argmin(np.abs(df['az']-180))]:.1f}°")
print()

# Check az range during session
print(f"Az range: {df['az'].min():.1f}° to {df['az'].max():.1f}°")
print(f"Alt range: {df['alt'].min():.1f}° to {df['alt'].max():.1f}°")
print(f"theta3 range: {df['theta3'].min():.1f}° to {df['theta3'].max():.1f}°")
print()

# Plot az, alt, theta3, and PA vs time
fig = make_subplots(rows=4, cols=1,
    subplot_titles=['Az (°)', 'Alt (°)', 'theta3 (°)', 'Parallactic Angle (°)'],
    vertical_spacing=0.06)

t_hrs = df['t_sec'] / 3600

for row, (y, name, color) in enumerate([
    (df['az'],    'az',    'royalblue'),
    (df['alt'],   'alt',   'orange'),
    (df['theta3'],'theta3','mediumseagreen'),
    (pa_deg,      'PA',    'violet')], 1):
    fig.add_trace(go.Scatter(x=t_hrs, y=y,
        mode='lines', line=dict(color=color, width=0.8),
        name=name), row=row, col=1)

fig.update_xaxes(title_text='Time (hours)', row=4, col=1)
fig.update_layout(height=800, width=1200,
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117',
    font=dict(color='#aaaacc'),
    title='Mount geometry during PEC session')
fig.show()